# 03 · Silver: skills mentioned in ads

Extracts which tools and skills each ad mentions, by matching a dictionary of regex patterns against the ad text.

- **Input:** `lh_silver.dbo.ads` (11,718 classified ads)
- **Output:** `lh_silver.dbo.ad_skills` — one row per ad and skill (bridge table)
- **How to run:** *Run all*. The table is overwritten, so re-running gives the same result.
- **Lakehouse:** `lh_silver` attached as default.

## Design notes

**Why a dictionary at all.** The structured fields `must_have.skills` and `nice_to_have.skills` are empty in every ad sampled — employers do not fill them in — so skills have to be read out of the free-text description.

**Headline + description, not description only.** Matching only the description produced a false finding: Power BI appeared to collapse from 42% of BI developer ads in 2019 to 5% in 2026. The cause is a change in how ads are written. Consultancy brokers publish framework-agreement ads titled e.g. "Senior Power BI-utvecklare, Region Skåne avrop 2688" with a short procurement text in the body. In 2019, 114 BI ads mentioned Power BI in the body and 1 had it only in the headline; in 2026 that is 7 and 37. Matching headline and description together puts Power BI back at ~30% of BI developer ads, which is plausible. This affects mainly the tools that appear in job titles; SQL went from 5,147 to 5,159 ads when headlines were added, Power BI from 996 to 1,157.

**R is the one skill that cannot be matched reliably.** A lone "r" appears throughout Swedish prose, matching 1,830 ads (16%) — clearly wrong. R is therefore only counted when it appears within the same sentence as another tool name (`python, sql, r, matlab`), which gives 1,427 ads. This undercounts ads that mention R alone and may include the occasional false positive.

**Word boundaries use lookarounds** rather than `\b`, so that `c#` works (`#` is not a word character) and `sql` matches inside "T-SQL" and "SQL-kunskaper" but not inside a longer word.

**What is being measured.** This counts what ads *say*, not what jobs actually require. 15% of ads mention no tool at all; inspecting them shows normal-length texts written in an employer-branding style ("we are looking for brilliant people…") rather than short stub ads — the most common words in them contain no tool names, so this is not a gap in the dictionary.

**Shares must be computed within role and year.** Comparing a skill against all ads is misleading, because the role mix changed: BI Developer was 33% of all ads in 2019 and 14% in 2025, which on its own makes BI tools look like they are declining. Year-to-year variation within a single role is also large (130–270 BI ads per year), so only long-term trends should be read.


## 1. Read the ads

Headline and description are concatenated with a full stop between them, so that the R pattern (which looks within a sentence) does not run the headline into the first sentence of the body.

In [2]:
from pyspark.sql import functions as F

SOURCE_TABLE = "lh_silver.dbo.ads"
TARGET_TABLE = "lh_silver.dbo.ad_skills"

ads = (spark.table(SOURCE_TABLE)
       .withColumn("text", F.lower(F.concat_ws(". ",
                   F.col("headline"),
                   F.coalesce(F.col("description"), F.lit("")))))
       .select("ad_id", "role", "region", "published_at", "text"))

n_ads = ads.count()
print("Ads:", n_ads)

StatementMeta(, ffe2ec97-8b17-4298-a258-692c1d34dfb3, 3, Finished, Available, Finished, False)

Ads: 11718


## 2. The skill dictionary

In [3]:
def b(p):
    """Word boundary that also works for characters like # and +."""
    return rf"(?<![a-z0-9åäö]){p}(?![a-z0-9åäö])"

# (skill, category, regex). Matched against headline + description, lower-cased.
SKILLS = [
    # Languages and analysis tools
    ("SQL",             "Language",    b("sql")),
    ("Python",          "Language",    b("python")),
    # R cannot be matched on its own (a lone "r" is everywhere in Swedish text),
    # so it is only counted when it appears next to another tool name. See design notes.
    ("R",               "Language",    r"(python|sql|sas|spss|matlab|stata|scala|java)[^.]{0,40}(?<![a-z0-9åäöé])r(?![a-z0-9åäöé])|(?<![a-z0-9åäöé])r(?![a-z0-9åäöé])[^.]{0,40}(python|sql|sas|spss)|\brstudio\b"),
    ("Scala",           "Language",    b("scala")),
    ("Java",            "Language",    b("java")),
    ("C#",              "Language",    b(r"c\#")),
    ("DAX",             "Language",    b("dax")),
    ("Excel",           "Analysis",    b("excel")),
    ("SAS",             "Analysis",    b("sas")),
    ("SPSS",            "Analysis",    b("spss")),
    ("Matlab",          "Analysis",    b("matlab")),
    ("Stata",           "Analysis",    b("stata")),
    # BI and visualisation
    ("Power BI",        "BI",          r"power\s*bi|powerbi"),
    ("Tableau",         "BI",          b("tableau")),
    ("Qlik",            "BI",          b("qlik(view|sense)?")),
    ("Looker",          "BI",          b("looker")),
    ("SSIS/SSAS/SSRS",  "BI",          b("ss(is|as|rs)")),
    ("Cognos",          "BI",          b("cognos")),
    # Cloud and platform
    ("Azure",           "Cloud",       b("azure")),
    ("AWS",             "Cloud",       r"\baws\b|amazon web services"),
    ("GCP",             "Cloud",       r"\bgcp\b|google cloud"),
    ("Microsoft Fabric","Cloud",       b("fabric")),
    ("Synapse",         "Cloud",       b("synapse")),
    ("Databricks",      "Cloud",       b("databricks")),
    ("Snowflake",       "Cloud",       b("snowflake")),
    # Data engineering
    ("Spark",           "Engineering", b("(py)?spark")),
    ("Hadoop",          "Engineering", b("hadoop")),
    ("Kafka",           "Engineering", b("kafka")),
    ("Airflow",         "Engineering", b("airflow")),
    ("dbt",             "Engineering", b("dbt")),
    ("ETL/ELT",         "Engineering", r"\be[tl]l\b|extract.{0,15}transform"),
    ("Data warehouse",  "Engineering", r"data\s*warehouse|datalager|\bdwh?\b"),
    ("Data modelling",  "Engineering", r"datamodell|data model|dimensionsmodell|star schema|kimball"),
    ("Docker",          "Engineering", b("docker")),
    ("Kubernetes",      "Engineering", r"\bkubernetes\b|\bk8s\b"),
    ("Git",             "Engineering", b("git(hub|lab)?")),
    ("Terraform",       "Engineering", b("terraform")),
    # Databases
    ("Oracle",          "Database",    b("oracle")),
    ("PostgreSQL",      "Database",    r"postgres(ql)?"),
    ("MongoDB",         "Database",    b("mongodb")),
    ("SAP",             "Database",    b("sap")),
    # Methods
    ("Machine learning","Method",      r"machine learning|maskininlärning|\bml\b"),
    ("Deep learning",   "Method",      r"deep learning|neurala nätverk|neural network"),
    ("NLP",             "Method",      r"\bnlp\b|natural language|språkteknologi"),
    ("Statistics",      "Method",      r"statistik|statistical|statistisk"),
    ("A/B testing",     "Method",      r"a/b[\s-]*test|ab[\s-]*test|hypotesprövning"),
]
print(len(SKILLS), "skills in the dictionary")

StatementMeta(, ffe2ec97-8b17-4298-a258-692c1d34dfb3, 4, Finished, Available, Finished, False)

46 skills in the dictionary


## 3. Match, and build one row per ad and skill

In [4]:
matches = F.array(*[
    F.when(F.col("text").rlike(pattern),
           F.struct(F.lit(skill).alias("skill"), F.lit(cat).alias("category")))
    for skill, cat, pattern in SKILLS
])

ad_skills = (ads
    .withColumn("m", F.explode(F.filter(matches, lambda x: x.isNotNull())))
    .select("ad_id", "role", "region", "published_at",
            F.col("m.skill").alias("skill"), F.col("m.category").alias("category")))

ad_skills.cache()
print("Skill matches:", ad_skills.count())

ad_skills.groupBy("skill").agg(
    F.countDistinct("ad_id").alias("ads"),
    F.round(F.countDistinct("ad_id") / F.lit(n_ads) * 100, 1).alias("pct_of_all_ads"),
).orderBy(F.desc("ads")).show(25)

StatementMeta(, ffe2ec97-8b17-4298-a258-692c1d34dfb3, 5, Finished, Available, Finished, False)

Skill matches: 46015
+----------------+----+--------------+
|           skill| ads|pct_of_all_ads|
+----------------+----+--------------+
|             SQL|5159|          44.0|
|          Python|4474|          38.2|
|Machine learning|3273|          27.9|
|  Data modelling|2573|          22.0|
|  Data warehouse|2566|          21.9|
|           Azure|2364|          20.2|
|         ETL/ELT|2269|          19.4|
|      Statistics|2001|          17.1|
|           Spark|1443|          12.3|
|             AWS|1433|          12.2|
|               R|1427|          12.2|
|        Power BI|1157|           9.9|
|            Java|1038|           8.9|
|            Qlik| 962|           8.2|
|             Git| 922|           7.9|
|             GCP| 920|           7.9|
|      Databricks| 882|           7.5|
|          Hadoop| 796|           6.8|
|           Scala| 795|           6.8|
|  SSIS/SSAS/SSRS| 744|           6.3|
|         Tableau| 721|           6.2|
|       Snowflake| 645|           5.5|
|   

## 4. Write the bridge table

In [5]:
(ad_skills.write
    .mode("overwrite")                      # idempotent: re-running replaces the table
    .option("overwriteSchema", "true")
    .saveAsTable(TARGET_TABLE))

t = spark.table(TARGET_TABLE)
n_rows = t.count()
n_with_skill = t.select("ad_id").distinct().count()
print(f"Rows: {n_rows}")
print(f"Ads with at least one skill: {n_with_skill} of {n_ads} ({n_with_skill / n_ads:.0%})")

assert n_rows > 0, "No skill matches written"
assert n_with_skill / n_ads > 0.7, "Suspiciously few ads matched any skill — check the dictionary"

StatementMeta(, ffe2ec97-8b17-4298-a258-692c1d34dfb3, 6, Finished, Available, Finished, False)

Rows: 46015
Ads with at least one skill: 9996 of 11718 (85%)


## 5. Results

The two questions the project set out to answer: how the roles differ, and how demand has changed. Both are computed as shares **within** role and year.

In [6]:
sk = spark.table(TARGET_TABLE)

# Skill profile per role
ads_per_role = ads.groupBy("role").count().withColumnRenamed("count", "n_role")
(sk.groupBy("role", "skill").agg(F.countDistinct("ad_id").alias("k"))
   .join(ads_per_role, "role")
   .withColumn("pct", F.round(F.col("k") / F.col("n_role") * 100, 1))
   .groupBy("skill").pivot("role").agg(F.first("pct"))
   .orderBy(F.desc("Data Engineer")).show(25))

StatementMeta(, ffe2ec97-8b17-4298-a258-692c1d34dfb3, 7, Finished, Available, Finished, False)

+----------------+------------------+------------+------------+-------------+--------------+
|           skill|Analytics Engineer|BI Developer|Data Analyst|Data Engineer|Data Scientist|
+----------------+------------------+------------+------------+-------------+--------------+
|          Python|              40.8|         8.0|        29.1|         47.4|          62.1|
|             SQL|              51.6|        53.2|        36.0|         46.5|          37.7|
|           Azure|              11.2|        19.0|         6.2|         32.8|          12.8|
|         ETL/ELT|              17.9|        29.5|         5.8|         28.7|           5.2|
|  Data modelling|              48.4|        31.4|        11.0|         28.1|           9.1|
|  Data warehouse|              24.7|        44.3|         6.7|         27.8|           2.9|
|Machine learning|              18.8|         6.9|        17.6|         26.6|          65.8|
|           Spark|               9.4|         0.8|         2.1|       

In [10]:
# Trend within role: share of that role's ads mentioning each skill
ads_ry = (ads.groupBy(F.year("published_at").alias("year"), "role").count()
             .withColumnRenamed("count", "n"))

trend = (sk.groupBy(F.year("published_at").alias("year"), "role", "skill")
   .agg(F.countDistinct("ad_id").alias("k"))
   .join(ads_ry, ["year", "role"])
   .withColumn("pct", F.round(F.col("k") / F.col("n") * 100, 1)))

print("Data Engineer — platform tools (% of that year's Data Engineer ads):")
(trend.filter("role = 'Data Engineer' and skill in ('Azure','AWS','Databricks','Snowflake','Spark','dbt','Microsoft Fabric')")
      .groupBy("year").pivot("skill").agg(F.first("pct")).orderBy("year").show())

print("BI Developer — BI tools:")
(trend.filter("role = 'BI Developer' and skill in ('Power BI','Qlik','Tableau','SSIS/SSAS/SSRS')")
      .groupBy("year").pivot("skill").agg(F.first("pct")).orderBy("year").show())

StatementMeta(, ffe2ec97-8b17-4298-a258-692c1d34dfb3, 11, Finished, Available, Finished, False)

Data Engineer — platform tools (% of that year's Data Engineer ads):
+----+----+-----+----------+----------------+---------+-----+----+
|year| AWS|Azure|Databricks|Microsoft Fabric|Snowflake|Spark| dbt|
+----+----+-----+----------+----------------+---------+-----+----+
|2016|13.3|  3.3|      NULL|            NULL|     NULL| 53.3|NULL|
|2017|13.2|  5.3|      NULL|            NULL|     NULL| 28.9|NULL|
|2018|17.1| 18.8|      NULL|            NULL|      1.7| 39.3|NULL|
|2019|21.3| 15.0|       3.9|            NULL|      3.9| 38.2|NULL|
|2020|30.2| 24.0|       4.1|            NULL|      5.0| 37.2|NULL|
|2021|17.9| 30.5|       6.0|            NULL|      5.5| 25.2| 2.9|
|2022|22.9| 32.1|      12.3|            NULL|      7.5| 22.3| 5.0|
|2023|18.1| 28.9|       9.4|             0.6|      5.6| 13.0| 6.6|
|2024|26.0| 39.5|      20.6|             5.4|     10.5| 20.8|11.6|
|2025|19.9| 35.4|      24.9|             6.5|     16.1| 21.8|15.9|
|2026|22.0| 45.8|      30.4|            14.8|     17.5| 24.3